<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Gymnasiet_Lab_2_Maskininlarning/Lektion_7_Battre_Utvardering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 Maskininlärning – Lektion 7: Bättre utvärdering och lösningar

**Målgrupp:** Gymnasiet, 16 år, inga förkunskaper krävs  
**Tid:** ca 50 minuter  
**Mål:** Lära sig hur man hanterar obalanserad data, förstå Precision och Recall, och se hur man förbättrar en AI

> 📋 **Förutsättning:** Du har gjort Lektion 5 och Lektion 6.

---

### Upphovspersoner
Originalversion: David Bergström & Mattias Tiger, mattias.tiger@liu.se  
Gymnasieversion baserad på originalverket ovan.

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/

---
## 🔁 Del 1 – Snabb repetition och problemet vi ska lösa

I Lektion 6 tränade vi en XGBoost-modell och såg att:
- Noggrannheten (Accuracy) var över 99% – men modellen missade ändå många bedrägerier
- Orsaken är **obalanserad data (Imbalanced Data)**: AI:n har tränat på mest normala köp

Idag lär vi oss **tre sätt** att angripa det här problemet:

1. **Vikta felklassificeringar:** Berätta för modellen att ett missat bedrägeri är mycket värre än ett falskt larm
2. **Bättre mätvärden:** Använd Precision och Recall istället för bara Accuracy
3. **Parameteroptimering (Hyperparameter Tuning):** Finjustera modellens inställningar

*Tänk dig ett fiskenät med stora hål – det fångar stor fisk men inte liten. Obalansproblemet liknar detta: AI:n är tränad att "fånga" normala köp (som är vanligast), men missar de sällsynta bedrägerifiskarna.  
Vi ska laga hålen i nätet!* 🐟🕳️

---
## 📥 Del 2 – Ladda in datan och förbered

Vi laddar in bedrägeridatan igen och delar upp den på samma sätt som i Lektion 6.

In [ ]:
!pip install xgboost -q
print("✅ XGBoost installerat!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, ConfusionMatrixDisplay,
                              precision_score, recall_score, classification_report)
from xgboost import XGBClassifier

print("Laddar data från internet... (kan ta 30-60 sekunder)")
url = 'https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/refs/heads/master/creditcard.csv'
df = pd.read_csv(url)

# Dela upp i träning och test (exakt samma som Lektion 6)
X = df.drop(columns=['Class'])
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n✅ Data laddad och uppdelad!")
print(f"   Träningsdata:  {len(X_train):,} transaktioner")
print(f"   Testdata:      {len(X_test):,} transaktioner")

---
## ⚖️ Del 3 – Hur obalanserat är datasetet?

Innan vi kan lösa problemet måste vi förstå hur *obalanserat* det faktiskt är.

Vi räknar hur många normala köp det finns per bedrägeri i träningsdatan.  
Det här talet blir vår **vikningsfaktor (Scale Weight)** – en nyckel till att förbättra modellen!

In [ ]:
# Räkna bedrägerier vs normala i träningsdatan
antal_bedragerier_train = (y_train == 1).sum()
antal_normala_train = (y_train == 0).sum()

print(f"Bedrägerier i träningsdata:    {antal_bedragerier_train:,}")
print(f"Normala köp i träningsdata:    {antal_normala_train:,}")
print()

# Beräkna vikningsfaktorn: hur många normala per bedrägeri?
vikningsfaktor = antal_normala_train / antal_bedragerier_train
print(f"Vikningsfaktor (scale_pos_weight): {vikningsfaktor:.1f}")
print()
print(f"Det finns ungefär {vikningsfaktor:.0f} normala transaktioner för varje bedrägeri.")
print("Vi ska nu berätta för AI:n att varje bedrägeri väger lika tungt som")
print(f"{vikningsfaktor:.0f} normala transaktioner!")

### Vad är vikningsfaktorn?

Tänk dig ett betygsätt på ett prov:  
- Om 99% av frågorna handlar om Historia och 1% om Matematik, och du missar Matematikfrågan – är det inte lika allvarligt!
- Men om Matematikfrågan är värd 100 gånger mer poäng, måste du plugga på Matematik extra!

Det är precis vad **vikningsfaktorn (scale_pos_weight)** gör:  
den säger till AI:n att ett missat bedrägeri väger lika tungt som ~570 normala transaktioner.

Utan den här vikten behandlar AI:n alla transaktioner lika, och lär sig mest från normala köp (som är vanligast).

---
## 🔧 Del 4 – Träna med vikningsfaktor (scale_pos_weight)

Nu tränar vi en ny modell – men den här gången berättar vi för AI:n hur viktig bedrägeri-klassen är.  
Det gör vi med parametern `scale_pos_weight` i XGBoost.

In [ ]:
print("Tränar förbättrad modell med vikningsfaktor...")
print("(kan ta 60-120 sekunder)")

# Viktat modell: scale_pos_weight = vikningsfaktorn vi beräknade
modell_viktad = XGBClassifier(
    max_depth=6,
    n_estimators=10,
    scale_pos_weight=vikningsfaktor,   # <-- Detta är den viktiga ändringen!
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
modell_viktad.fit(X_train, y_train)

y_pred_viktad = modell_viktad.predict(X_test)

print(f"\n✅ Förbättrad modell tränad!")
print(f"   Noggrannhet (Accuracy): {accuracy_score(y_test, y_pred_viktad):.2%}")

---
## 📊 Del 5 – Jämför: Ovitkad vs. viktad modell

Låt oss se hur förväxlingsmatriserna ser ut nu!  
Vi jämför den gamla modellen (utan vikter) mot den nya förbättrade modellen.

In [ ]:
# Träna om den gamla modellen (utan vikter) för jämförelse
modell_gammal = XGBClassifier(
    max_depth=6, n_estimators=10,
    random_state=42, eval_metric='logloss', verbosity=0
)
modell_gammal.fit(X_train, y_train)
y_pred_gammal = modell_gammal.predict(X_test)

# Jämför förväxlingsmatriser
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gammal modell
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_gammal,
    display_labels=['Normalt', 'Bedrägeri'],
    colorbar=False, ax=axes[0]
)
axes[0].set_title(f'Utan vikter\n(Accuracy: {accuracy_score(y_test, y_pred_gammal):.2%})', fontsize=11)

# Ny viktad modell
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_viktad,
    display_labels=['Normalt', 'Bedrägeri'],
    colorbar=False, ax=axes[1]
)
axes[1].set_title(f'Med vikningsfaktor (scale_pos_weight)\n(Accuracy: {accuracy_score(y_test, y_pred_viktad):.2%})', fontsize=11)

plt.suptitle('Förbättring med vikningsfaktor – Förväxlingsmatriser', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Räkna resultaten för båda modellerna
totalt = y_test.sum()

tp_gammal = ((y_pred_gammal == 1) & (y_test == 1)).sum()
fn_gammal = ((y_pred_gammal == 0) & (y_test == 1)).sum()
fp_gammal = ((y_pred_gammal == 1) & (y_test == 0)).sum()

tp_viktad = ((y_pred_viktad == 1) & (y_test == 1)).sum()
fn_viktad = ((y_pred_viktad == 0) & (y_test == 1)).sum()
fp_viktad = ((y_pred_viktad == 1) & (y_test == 0)).sum()

print(f"Totalt antal bedrägerier i testdata: {totalt}")
print()
print("=" * 55)
print(f"{'':35} {'Utan vikter':>8}  {'Med vikter':>8}")
print("=" * 55)
print(f"✅ Hittade bedrägerier (True Positive):  {tp_gammal:>8}  {tp_viktad:>8}")
print(f"   (% av alla bedrägerier):              {tp_gammal/totalt:>7.1%}  {tp_viktad/totalt:>7.1%}")
print(f"❌ Missade bedrägerier (False Negative): {fn_gammal:>8}  {fn_viktad:>8}")
print(f"⚠️  Falska larm (False Positive):        {fp_gammal:>8}  {fp_viktad:>8}")

### 💬 Reflektionsfråga 7.1

Titta på jämförelsen ovan.

**Fråga 1:** Hittade den viktade modellen fler eller färre bedrägerier?

**Fråga 2:** Vad hände med antalet falska larm (Falskt Positiv)? Ökade eller minskade det?

**Fråga 3:** Noggrannheten (Accuracy) sjönk förmodligen lite – men är det ett problem?  
*(Tips: Tänk på vad vi lärde oss i Lektion 5 om noggrannhetsfällan!)*

---
## 📐 Del 6 – Bättre mätvärden: Precision och Recall

Nu lär vi oss två mycket viktigare mätvärden för obalanserade problem:

### 🎯 Recall (Känslighet / Återkallelse)
**Av alla riktiga bedrägerier – hur många hittade AI:n?**

```
Recall = Sant Positiv (TP) / (Sant Positiv (TP) + Falskt Negativ (FN))
       = Hittade bedrägerier / Alla bedrägerier
```

Tänk dig en metalldetektor på en flygplats:
- **Hög Recall** = Detektorn piper vid nästan alla föremål → Missar sällan något farligt
- **Låg Recall** = Detektorn missar många föremål → Farligt!

---

### 🔍 Precision (Exakthet / Noggrannhet för positiva)
**Av alla gånger AI:n sa "Bedrägeri" – hur ofta hade den rätt?**

```
Precision = Sant Positiv (TP) / (Sant Positiv (TP) + Falskt Positiv (FP))
           = Rätt bedrägerivarningar / Alla bedrägerivarningar
```

Återgår vi till flygplatsen:
- **Hög Precision** = När detektorn piper är det nästan alltid farligt → Få falska larm
- **Låg Precision** = Detektorn piper vid allt (nycklar, bälten, etc.) → Massor av falska larm

---

### ⚖️ Avvägningen (Trade-off)
Det finns en **spänning** mellan Precision och Recall:
- Om vi vill stoppa *alla* bedrägerier (hög Recall) → Fler falska larm (låg Precision)
- Om vi vill ha *mycket säker* varning (hög Precision) → Missar vi fler bedrägerier (låg Recall)

**Det finns inget perfekt svar!** Det beror på vad som är viktigast i just den situationen.

In [ ]:
# Beräkna Precision och Recall för båda modellerna
precision_gammal = precision_score(y_test, y_pred_gammal)
recall_gammal = recall_score(y_test, y_pred_gammal)

precision_viktad = precision_score(y_test, y_pred_viktad)
recall_viktad = recall_score(y_test, y_pred_viktad)

print("=" * 55)
print(f"{'Mätvärde':35} {'Utan vikter':>8}  {'Med vikter':>8}")
print("=" * 55)
print(f"Noggrannhet (Accuracy):              {accuracy_score(y_test, y_pred_gammal):>8.2%}  {accuracy_score(y_test, y_pred_viktad):>8.2%}")
print(f"Precision:                           {precision_gammal:>8.2%}  {precision_viktad:>8.2%}")
print(f"Recall (Känslighet):                 {recall_gammal:>8.2%}  {recall_viktad:>8.2%}")
print()
print("Tolkning av Recall:")
print(f"  Utan vikter: AI:n hittar {recall_gammal:.1%} av alla bedrägerier")
print(f"  Med vikter:  AI:n hittar {recall_viktad:.1%} av alla bedrägerier")

### 💬 Reflektionsfråga 7.2

Titta på Precision och Recall för de två modellerna.

**Fråga 1:** Vilken modell har bäst Recall? Vad betyder det i praktiken?

**Fråga 2:** Vilken modell skulle du välja om du var chef på en bank som prioriterar:
- *Att stoppa alla bedrägerier, även om det kostar falska larm*
- *Att aldrig störa en oskyldigs köp, men kanske missa lite bedrägerier*

**Fråga 3:** Tänk dig en annan situation – ett cancertest (som vi diskuterade i Lektion 5).  
Vad borde prioriteras: Hög Recall eller Hög Precision? Varför?

---
## 🔬 Del 7 – Finjustera modellen: Parameteroptimering (Hyperparameter Tuning)

Vi har redan förbättrat modellen med vikningsfaktorn.  
Men det finns ytterligare ett steg: **Parameteroptimering (Hyperparameter Tuning)**.

Tänk dig en ny bilmotor. Den har massor av inställningar:
- Hur mycket bränsle?
- Vilken kompressionsnivå?
- Tändningstid?

En erfaren mekaniker justerar alla dessa för att maximera prestandan.  
På samma sätt kan AI-ingenjörer justera modellens "inre inställningar":

| Parameter | Vad det styr | Exempel |
|-----------|-------------|---------|
| `max_depth` | Hur djupa beslutsträden är | 1 (enkelt) → 10 (komplext) |
| `n_estimators` | Hur många träd i "skogen" | 1 → 1000 |
| `eta` (learning rate) | Hur snabbt AI:n lär sig | 0.01 (försiktig) → 0.3 (snabb) |
| `scale_pos_weight` | Hur viktigt det är att hitta bedrägerier | 1 (neutralt) → 570+ (starkt viktad) |

I avancerade projekt använder ingenjörer automatiska verktyg (som *Bayesiansk optimering*) för att systematiskt testa hundratals kombinationer och hitta de bästa inställningarna. Det är som att ha en robot som provar alla möjliga motorinställningar!

I den ursprungliga laborationen visas ett exempel på detta med ett verktyg kallat `hyperopt`.  
Det kräver mer avancerad kod, men principen är enkel:

> **Definiera vad "bra" betyder** (t.ex. minimera antalet Falskt Negativ)  
> → **Prova många kombinationer** av parametrar  
> → **Välj den bästa kombinationen**

Vi kan prova en enkel manuell jämförelse för att se effekten av att ändra ett par parametrar:

In [ ]:
# Enkel manuell parameteroptimering
# Vi testar tre olika inställningar och ser vilken som är bäst för Recall

konfigurationer = [
    {'max_depth': 3,  'n_estimators': 5,  'scale_pos_weight': vikningsfaktor},
    {'max_depth': 6,  'n_estimators': 10, 'scale_pos_weight': vikningsfaktor},
    {'max_depth': 6,  'n_estimators': 20, 'scale_pos_weight': vikningsfaktor * 1.5},
]

print("Testar olika konfigurationer...")
print()
print(f"{'Konfiguration':>15} {'max_depth':>10} {'n_est.':>8} {'scale_w':>8} {'Recall':>8} {'Precision':>10} {'Accuracy':>10}")
print("-" * 75)

basta_recall = 0
basta_konfiguration = None
basta_pred = None

for i, config in enumerate(konfigurationer):
    modell = XGBClassifier(
        max_depth=config['max_depth'],
        n_estimators=config['n_estimators'],
        scale_pos_weight=config['scale_pos_weight'],
        random_state=42, eval_metric='logloss', verbosity=0
    )
    modell.fit(X_train, y_train)
    y_pred = modell.predict(X_test)

    r = recall_score(y_test, y_pred)
    p = precision_score(y_test, y_pred)
    a = accuracy_score(y_test, y_pred)

    marker = " ← bäst" if r > basta_recall else ""
    print(f"  Konfiguration {i+1:>1}  {config['max_depth']:>10}  {config['n_estimators']:>6}  {config['scale_pos_weight']:>8.1f}  {r:>7.2%}  {p:>9.2%}  {a:>9.2%}{marker}")

    if r > basta_recall:
        basta_recall = r
        basta_konfiguration = config
        basta_pred = y_pred

print()
print(f"✅ Bästa konfiguration: max_depth={basta_konfiguration['max_depth']}, n_estimators={basta_konfiguration['n_estimators']}")

---
## 🏁 Del 8 – Slutlig jämförelse: Alla modeller

Nu jämför vi alla tre modeller sida vid sida:
1. Utan vikter (original)
2. Med vikningsfaktor
3. Med bästa konfiguration

In [ ]:
# Slutlig visualisering av den bästa konfigurationen
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Modell utan vikter (referens)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_gammal,
    display_labels=['Normalt', 'Bedrägeri'],
    colorbar=False, ax=axes[0]
)
axes[0].set_title(
    f'Utan vikter (referens)\n'
    f'Recall: {recall_score(y_test, y_pred_gammal):.1%} | '
    f'Precision: {precision_score(y_test, y_pred_gammal):.1%}',
    fontsize=10
)

# Bästa konfiguration
ConfusionMatrixDisplay.from_predictions(
    y_test, basta_pred,
    display_labels=['Normalt', 'Bedrägeri'],
    colorbar=False, ax=axes[1]
)
axes[1].set_title(
    f'Bästa konfiguration (med vikter)\n'
    f'Recall: {recall_score(y_test, basta_pred):.1%} | '
    f'Precision: {precision_score(y_test, basta_pred):.1%}',
    fontsize=10
)

plt.suptitle('Slutlig jämförelse – Referens vs. Bästa modell', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== Sammanfattande jämförelse ===")
print(f"{'Modell':35} {'Recall':>8}  {'Precision':>10}  {'Accuracy':>10}")
print("-" * 70)
for namn, y_pred in [
    ("Utan vikter (referens)", y_pred_gammal),
    ("Med vikningsfaktor", y_pred_viktad),
    ("Bästa konfiguration", basta_pred),
]:
    print(f"{namn:35} {recall_score(y_test, y_pred):>8.2%}  {precision_score(y_test, y_pred):>10.2%}  {accuracy_score(y_test, y_pred):>10.2%}")

---
## 💬 Del 9 – Stor diskussion: Reflektera och utvärdera

Nu när du sett hela processen – från att ladda data till att optimera en AI – är det dags att tänka djupare.

### Diskussionsfråga 1: Vad är bättre?
Vi kan nu se att vi kan förbättra Recall (hitta fler bedrägerier) men att det kostar Precision (fler falska larm).

Är det bättre att:
- **A:** Stoppa fler bedrägerier (högre Recall) men råka spärra fler oskyldigas kort (lägre Precision)?
- **B:** Nästan aldrig störa en oskyldig kund (högre Precision) men missa fler bedrägerier (lägre Recall)?

**Svar:** Det beror på kontexten! Diskutera:
- Hur tror du en bank väger detta beslut?
- Vad händer om AI:n piper larm på allt – folk slutar lita på den!
- Vad händer om AI:n inte piper tillräckligt – bedragare slipper alltid igenom!

---

### Diskussionsfråga 2: Vilka slutsatser drar du?

Titta på alla resultat vi sett:

1. Kan man se **bara på noggrannhet (Accuracy)** om en bedrägerimodell är bra?  
   *(Tips: Vad lärde vi oss i Lektion 5?)*

2. Vad är det viktigaste affärsvärdet för en bank:
   - Att ha 99,9% accuracy (men missa 70% av bedrägerier)?
   - Att ha 98% accuracy (men bara missa 30% av bedrägerier)?

3. **Den stora lärdomen:** Maskininlärning handlar inte bara om att skriva kod – det handlar om att  
   *förstå problemet*, *välja rätt mätvärden* och *fatta etiska beslut*.

---

### Diskussionsfråga 3: Etik och AI
AI-modeller kan ibland diskriminera utan avsikt. Tänk dig att bedrägerier är vanligare i vissa regioner  
och att AI:n lärt sig detta mönster.

- Är det etiskt att blockera köp från dessa regioner oftare?  
- Vad är skillnaden mellan att modellen "hittar ett verkligt mönster" och att den är "diskriminerande"?
- Vem bär ansvaret för AI:ns beslut – programmeraren, banken, eller kunden?

---
## 🎓 Grattis – du har klarat alla 7 lektioner! 🎉

### Sammanfattning av hela kursen

| Begrepp | Förklaring | Engelskt namn |
|---------|------------|---------------|
| **Övervakad inlärning** | AI lär sig från märkta exempel | Supervised Learning |
| **Egenskaper** | Det AI:n mäter/analyserar | Features |
| **Målvariabel** | Det AI:n ska förutsäga | Target / Label |
| **Träningsdata** | Data AI:n lär sig av | Training Data |
| **Testdata** | Data AI:n utvärderas på | Test Data |
| **Modell** | AI:ns "inlärda mönster" | Model |
| **Beslutsträd** | Modell som ställer ja/nej-frågor | Decision Tree |
| **Noggrannhet** | Andel rätta svar totalt | Accuracy |
| **Förväxlingsmatris** | Rutnät som visar alla fyra typer av svar | Confusion Matrix |
| **Sant Positiv** | Bedrägeri korrekt flaggat | True Positive (TP) |
| **Falskt Negativ** | Missad detektion – farligt! | False Negative (FN) |
| **Falskt Positiv** | Falskt larm | False Positive (FP) |
| **Obalanserad data** | En kategori är extremt vanligare | Imbalanced Data |
| **Vikningsfaktor** | Säger åt AI:n hur viktig sällsynt klass är | scale_pos_weight |
| **Recall (Känslighet)** | Andel riktiga bedrägerier som hittades | Recall / Sensitivity |
| **Precision (Exakthet)** | Andel bedrägerievarningar som stämde | Precision |
| **Parameteroptimering** | Finjustering av modellens inställningar | Hyperparameter Tuning |

---

### Den stora insikten 🌟

Maskininlärning handlar inte bara om att skriva kod –  
det handlar om att:

1. 🎯 **Förstå problemet** – Vad vill vi egentligen lösa?
2. 📏 **Välja rätt mätvärden** – Accuracy räcker inte alltid!
3. ⚖️ **Fatta etiska beslut** – AI är ett verktyg som kan göra skada om det används fel
4. 🔄 **Iterera och förbättra** – Maskininlärning är en process, inte en engångslösning

AI är ett kraftfullt verktyg, men det är alltid **människor** som bestämmer hur det används!

---

*"Med stor kraft kommer stort ansvar."* – Det gäller för AI precis som för allt annat. 🤖